In [ ]:
import pandas as pd
import pickle
import arviz as az
from plotly import graph_objects as go
from datetime import datetime
from emu_renewal.inputs import DATA_PATH, MOB_MAP
from emu_renewal.outputs import get_output_dir, load_targets
from emu_renewal.plotting import plot_spaghetti_calib_comparison, plot_post_prior_comparison, plot_imm_props

In [ ]:
country = "France"
analysis = "mob"
time = "20250113_1545"

In [ ]:
outdir = get_output_dir(country, analysis, time)

In [ ]:
spaghetti = pd.read_hdf(outdir / "spaghetti.h5")
targets = load_targets(country, analysis, time)
idata = az.from_netcdf(outdir / "idata.nc")
with open(outdir / "description.txt") as file:
    print(file.read())

In [ ]:
mob = pd.read_csv(DATA_PATH / f"mobility/{MOB_MAP[country]}_mob_data.csv", index_col=0)
mob.index = pd.to_datetime(mob.index)
non_resi_mob = mob.loc[:, mob.columns!="residential"]
model_mob = non_resi_mob.mean(axis=1).rolling(7).mean().dropna()
analysis_start = datetime(2020, 9, 1)
analysis_end = datetime(2021, 6, 15)
filtered_mob = model_mob.loc[(analysis_start < model_mob.index) & (model_mob.index < analysis_end)]

In [ ]:
outputs = list(targets.keys()) + ["process", "seropos"]
spagh_plot = plot_spaghetti_calib_comparison(spaghetti, targets, outputs)
spagh_plot.add_traces(go.Scatter(x=filtered_mob.index, y=filtered_mob, line={"color": "blue"}), rows=outputs.index("process") + 1, cols=1)

In [ ]:
priors = pickle.load(open(outdir / "priors.pkl", "rb"))

In [ ]:
priors.keys()

In [ ]:
plot_post_prior_comparison(idata, ["gen_mean", "gen_sd"], priors);

In [ ]:
plot_post_prior_comparison(idata, ["cdr", "har", "ifr", "alpha_relinfect"], priors);

In [ ]:
delay_dist_keys = ["report_mean", "report_sd", "admit_mean", "admit_sd", "stay_mean", "stay_sd"]
plot_post_prior_comparison(idata, delay_dist_keys, priors, req_grid=[3, 2]);

In [ ]:
plot_post_prior_comparison(idata, ["shared_dispersion"], priors);

In [ ]:
plot_imm_props(spaghetti, 2)